# Proyecto 3: Mejorando algoritmos con CUDA

**CC3069 Computación Paralela y Distribuida, UVG.** Fernando Mendoza, Hansel y Adrian.

Partimos del Lab 3 de Data Science: una MobileNetV3-Small que reconoce las 29 señas del alfabeto ASL
(26 letras, `del`, `nothing` y `space`) a partir de fotos de 200 × 200. En CPU, su entrenamiento completo
tomó 3.7 horas. En este notebook mostramos la versión original y la nueva, con la separación que pide el
enunciado: la **preparación de las imágenes en los núcleos del CPU** (fase 1) y **la transformación de
entrada y el entrenamiento en la GPU con CUDA** (fase 2).

El código no se reimplementa aquí: todo se importa del paquete `asl_cuda` (`src/asl_cuda/`), que es el
mismo que usan los scripts de medición. Las celdas de GPU se saltan solas si no hay una GPU NVIDIA.

| Integrante | Parte |
|---|---|
| Fernando Mendoza | Análisis del Lab 3, diseño del algoritmo paralelo, kernel CUDA e implementación |
| Hansel | Medición de la versión secuencial y CPU, pruebas con datos nuevos, notebook e informe |
| Adrian | Corridas en GPU, entrenamiento final, evidencia de hardware y demo |

## 1. Entorno

Primero registramos en qué hardware corre el notebook. `asl_cuda.envinfo` es lo mismo que guarda cada
corrida en `environment.json`.

In [ ]:
import inspect
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, Markdown, display

from asl_cuda.constants import CLASSES, ROOT
from asl_cuda.envinfo import environment

GPU = torch.cuda.is_available()
device = torch.device("cuda" if GPU else "cpu")
env = environment(device)
print(f"CPU: {env['cpu']} | núcleos físicos {env['physical_cores']}, lógicos {env['logical_cores']} | RAM {env['ram_gb']} GB")
print(f"GPU: {env['gpu'].get('name', 'no disponible')}")
print("Versiones:", {k: v for k, v in env["packages"].items() if v})

In [ ]:
if GPU:
    !nvidia-smi

## 2. Datos y partición

El dataset `grassknoted/asl-alphabet` tiene 3,000 frames de video por clase. Frames vecinos son casi
idénticos, así que la partición no es al azar: se agrupan en ventanas de 100 frames y cada ventana va
completa a una sola partición (entrenamiento, validación o prueba). `asl-cuda prep` también comprueba con
SHA-256 que no haya archivos idénticos entre particiones.

In [ ]:
from asl_cuda.config import load_data_config, load_train_config

dcfg = load_data_config(ROOT / "configs/data.yaml")
tcfg = load_train_config(ROOT / "configs/train.yaml")
cache_dir = dcfg.path("cache_dir")
manifest = pd.read_csv(cache_dir / "manifest.csv")
conteos = manifest.groupby("split").agg(imagenes=("path", "size"), ventanas=("window", "nunique"))
conteos["por_clase"] = conteos["imagenes"] // len(CLASSES)
conteos

In [ ]:
from asl_cuda.prep import load_split

train_data = load_split(cache_dir, "train", dcfg.cache_size)
primeros = [int(np.flatnonzero(train_data.labels == c)[len(np.flatnonzero(train_data.labels == c)) // 2]) for c in range(len(CLASSES))]
fig, axes = plt.subplots(3, 10, figsize=(12, 4))
for ax, c, i in zip(axes.flat, range(len(CLASSES)), primeros):
    ax.imshow(train_data.images[i])
    ax.set_title(CLASSES[c], fontsize=9)
for ax in axes.flat:
    ax.axis("off")
fig.suptitle("Un ejemplo por clase, leído de la caché uint8", x=0.01, ha="left")
plt.tight_layout()

## 3. Versión original (secuencial)

Copiamos del Lab 3 la parte que se ejecuta en cada época: el `Dataset` que abre y decodifica cada JPEG y
la cadena de transformaciones de torchvision con PIL. El único cambio es que el ciclo de entrenamiento
mide por separado el tiempo de datos y el de cálculo.

In [ ]:
from asl_cuda import secuencial

print(inspect.getsource(secuencial.lab3_transform))
print(inspect.getsource(secuencial.FileDataset))

Medimos dos lotes de 64 imágenes en un solo hilo y sin procesos auxiliares, que es la versión secuencial
de referencia. Una medición de dos lotes es solo ilustrativa; las tablas oficiales (sección 7) usan 1,160
imágenes, calentamiento y repeticiones.

In [ ]:
from torch import nn

from asl_cuda.manifest import absolute_paths, subsample
from asl_cuda.model import build_model

muestra = subsample(manifest[manifest["split"] == "train"], 40).reset_index(drop=True)
rutas = absolute_paths(muestra)

hilos_antes = torch.get_num_threads()
torch.set_num_threads(1)
loader = secuencial.make_loader(muestra, rutas, tcfg.image_size, 64, augment=True, workers=0, seed=0, shuffle=True)
modelo = build_model(tcfg.architecture, pretrained=False)
optim = torch.optim.AdamW(modelo.parameters(), lr=1e-4)
r = secuencial.run_epoch(modelo, loader, optim, nn.CrossEntropyLoss(), clip=1.0, max_batches=2)
torch.set_num_threads(hilos_antes)

print(f"{r['images']} imágenes en {r['seconds']:.2f} s ({r['images'] / r['seconds']:.1f} img/s)")
print(f"  datos (abrir, decodificar, transformar): {r['data_seconds']:.2f} s ({100 * r['data_seconds'] / r['seconds']:.0f} %)")
print(f"  red (forward, backward, AdamW):          {r['compute_seconds']:.2f} s ({100 * r['compute_seconds'] / r['seconds']:.0f} %)")

## 4. Fase 1 · Preparación de imágenes en el CPU

Leer un archivo y decodificar un JPEG necesita el sistema de archivos y libjpeg, y la decodificación
Huffman es secuencial dentro de cada archivo. Lo que sí es paralelo es que los archivos son independientes
entre sí: `decode_to_npy` reparte bloques de rutas entre procesos y cada proceso escribe sus imágenes
directamente en un arreglo `uint8` en disco abierto con *memory map*. Esto se hace **una sola vez**; el
Lab 3 lo repetía en cada época (1,911,100 decodificaciones para 55,100 archivos).

Usamos procesos y no hilos por el GIL de Python (sección 2.4 del informe).

In [ ]:
import os
import tempfile

from asl_cuda.prep import decode_to_npy

muestra_prep = subsample(manifest[manifest["split"] == "train"], 200)
rutas_prep = absolute_paths(muestra_prep)

tiempos = {}
with tempfile.TemporaryDirectory() as tmp:
    # una pasada de calentamiento para que los JPEG ya estén en la caché del sistema operativo
    decode_to_npy(rutas_prep, Path(tmp) / "calentamiento.npy", dcfg.cache_size, 0)
    for workers in (1, 0):  # 0 = todos los núcleos lógicos
        t = time.perf_counter()
        stats, _ = decode_to_npy(rutas_prep, Path(tmp) / f"prueba_{workers}.npy", dcfg.cache_size, workers)
        tiempos[stats.workers] = time.perf_counter() - t

base = tiempos[1]
for procesos, segundos in tiempos.items():
    print(f"{procesos:>2} proceso(s): {segundos:.2f} s, {len(rutas_prep) / segundos:,.0f} img/s, speedup {base / segundos:.2f}")

## 5. Fase 2 · Kernel CUDA de entrada

La transformación de entrada (recorte, rotación, zoom, perspectiva, color, normalización y borrado) se
expresa como una homografía y 7 números más por imagen. Con eso cada píxel de salida es independiente:
**un hilo por píxel**, bloques de 16 × 16 = 256 hilos (8 warps) y un grid de 14 × 14 × B, donde la
tercera dimensión es la imagen dentro del lote.

La misma función por píxel (`_pixel`) se compila como función de dispositivo CUDA y como función de CPU
con `numba.njit(parallel=True)`.

In [ ]:
from asl_cuda import augment

print(inspect.getsource(augment._pixel))
print(inspect.getsource(augment._warp_kernel))
grid, bloque = augment.grid_for(64, tcfg.image_size, tcfg.image_size, (16, 16))
print(f"Lote de 64: grid {grid}, bloque {bloque} -> {np.prod(grid):,} bloques, {np.prod(grid) * np.prod(bloque):,} hilos")

Comprobamos que el kernel calcula lo mismo que una implementación NumPy independiente, y en GPU que
coincide con ella dentro del error de redondeo de float32.

In [ ]:
rng = np.random.default_rng(0)
lote = train_data.images[rng.choice(len(train_data), 16, replace=False)]
params = augment.sample_params(rng, len(lote), dcfg.cache_size, tcfg.image_size, tcfg.augment, train=True).packed()

referencia = augment.warp_numpy(lote, params, tcfg.image_size)
salida_cpu = augment.warp_cpu(lote, params, np.empty_like(referencia))
print(f"Numba CPU vs NumPy: diferencia máxima {np.abs(salida_cpu - referencia).max():.2e}")
print(f"(valores normalizados entre {referencia.min():.2f} y {referencia.max():.2f}; la diferencia es redondeo de float32)")

if GPU:
    from numba import cuda

    d_out = cuda.device_array(referencia.shape, np.float32)
    augment.launch_warp(cuda.to_device(lote), cuda.to_device(params), cuda.to_device(augment.IMAGENET_MEAN),
                        cuda.to_device(augment.IMAGENET_STD), d_out, (16, 16))
    salida_gpu = d_out.copy_to_host()
    print(f"CUDA vs NumPy:      diferencia máxima {np.abs(salida_gpu - referencia).max():.2e}")

In [ ]:
vista = salida_gpu if GPU else salida_cpu
img = vista.transpose(0, 2, 3, 1) * augment.IMAGENET_STD + augment.IMAGENET_MEAN
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, im in zip(axes.flat, img):
    ax.imshow(np.clip(im, 0, 1))
    ax.axis("off")
fig.suptitle("Salida del kernel: 16 imágenes con la aumentación de entrenamiento", x=0.01, ha="left")
plt.tight_layout()

## 6. Fase 2 · Entrenamiento en la GPU

`CudaFeeder` (`feeders.py`) arma cada lote en el CPU copiando 64 imágenes `uint8` de la caché a un buffer
*pinned*, lo copia a la GPU de forma asíncrona y lanza el kernel en el mismo stream que PyTorch; el
tensor resultante ya está en la GPU cuando empieza el forward. Con dos buffers alternados, el CPU prepara
el lote siguiente mientras la GPU entrena con el actual. Este es el paso de entrenamiento:

In [ ]:
from asl_cuda import train

print(inspect.getsource(train.train_step))

Para que se vea correr, entrenamos 10 lotes con el mismo feeder que usa el entrenamiento completo
(en GPU si está disponible; si no, con el kernel de Numba en CPU).

In [ ]:
from asl_cuda.model import set_trainable

train.configure_backend(device, tcfg)
amp = tcfg.cuda.amp and GPU
modelo = build_model(tcfg.architecture, pretrained=False).to(device)
if GPU:
    modelo = modelo.to(memory_format=torch.channels_last)
set_trainable(modelo, 4)
optim = torch.optim.AdamW((p for p in modelo.parameters() if p.requires_grad), lr=3e-4)
scaler = torch.amp.GradScaler(device.type, enabled=amp)
feeder = train.make_feeder(train_data, tcfg, device, train=True, limit=640)

t = time.perf_counter()
for paso, (x, y) in enumerate(feeder, 1):
    _, perdida = train.train_step(modelo, x, y, nn.CrossEntropyLoss(), optim, scaler, device, amp, 1.0)
if GPU:
    torch.cuda.synchronize()
segundos = time.perf_counter() - t
print(f"{paso} lotes en {segundos:.2f} s en {device} ({640 / segundos:.0f} img/s, incluye calentamiento), pérdida final {perdida.item():.3f}")
print(f"Tiempo armando lotes en el CPU: {feeder.data_wait:.2f} s")

## 7. Mediciones

Las mediciones oficiales vienen de `asl-cuda bench` en la laptop con GPU (CPU y GPU en la misma máquina,
para que el speedup tenga sentido). Aquí solo se cargan los resultados guardados en `results/`.

In [ ]:
def buscar(condicion):
    for carpeta in sorted((ROOT / "results").glob("*")):
        if condicion(carpeta):
            return carpeta
    return None

oficial = buscar(lambda c: (c / "train" / "base-cuda" / "summary.json").exists())
referencias_cpu = [c for c in sorted((ROOT / "results").glob("*")) if (c / "bench" / "environment.json").exists()
                  and not json.loads((c / "bench" / "environment.json").read_text()).get("gpu", {}).get("available")]
print("Corrida oficial (GPU):", oficial.name if oficial else "todavía no está en results/")
print("Computadoras de referencia (solo CPU):", ", ".join(c.name for c in referencias_cpu) or "ninguna")


def tabla(carpeta, nombre, titulo):
    ruta = carpeta / "tablas" / f"{nombre}.md" if carpeta else None
    if ruta and ruta.exists():
        display(Markdown(f"**{titulo}** (`{ruta.relative_to(ROOT)}`)\n\n" + ruta.read_text(encoding="utf-8")))


def figura(carpeta, nombre, ancho=720):
    ruta = carpeta / "figuras" / f"{nombre}.png" if carpeta else None
    if ruta and ruta.exists():
        display(Image(filename=str(ruta), width=ancho))

In [ ]:
tabla(oficial, "fases", "Resumen por fase")
figura(oficial, "speedup_fases")
tabla(oficial, "pipeline", "Época completa")
tabla(oficial, "proyeccion_entrenamiento", "Proyección del entrenamiento completo")

In [ ]:
tabla(oficial, "prep", "Fase 1: decodificación en CPU")
figura(oficial, "prep_escalamiento", 560)
tabla(oficial, "augment", "Transformación de entrada")
tabla(oficial, "kernel_bloques", "Kernel según hilos por bloque")
figura(oficial, "kernel_bloques", 480)
tabla(oficial, "train", "Paso de entrenamiento")
tabla(oficial, "inference", "Inferencia")

In [ ]:
tabla(oficial, "entrenamientos", "Entrenamientos completos")
figura(oficial, "entrenamiento_base-cuda")
figura(oficial, "monitor_bench")
figura(oficial, "monitor_base-cuda")

Las computadoras sin GPU sirven para ver cómo cambia el escalamiento en CPU con otra arquitectura.

In [ ]:
for carpeta in referencias_cpu:
    tabla(carpeta, "fases", f"Resumen por fase en {carpeta.name}")
    tabla(carpeta, "prep", f"Decodificación en {carpeta.name}")
    figura(carpeta, "monitor_bench")

## 8. Datos nuevos

El modelo se probó con el test interno (11,600 imágenes que nunca vio), con un dataset externo de 870 fotos
de otras manos y fondos (una mitad para elegir el modelo y otra para reportar) y con fotos del equipo.

In [ ]:
for carpeta in [oficial, *referencias_cpu]:
    if carpeta:
        tabla(carpeta, "evaluacion", f"Evaluación ({carpeta.name})")
if oficial and (oficial / "eval" / "seleccion_modelo.json").exists():
    print(json.dumps(json.loads((oficial / "eval" / "seleccion_modelo.json").read_text()), indent=2, ensure_ascii=False))

## 9. Conclusiones

El tiempo del Lab 3 tenía dos causas distintas y cada una tuvo su solución en un procesador diferente. La
preparación de archivos era trabajo repetido: decodificar los mismos JPEG en cada época. Ahora se hace una
sola vez, repartida entre los procesos del CPU, y queda en una caché `uint8`. La transformación de entrada y
la red son aritmética regular sobre tensores, y eso es lo que pasó a la GPU: un kernel propio en Numba CUDA,
con un hilo por píxel, y el entrenamiento con cuDNN en precisión mixta. Las dos fases se comunican con
copias asíncronas de `uint8`, cinco veces más pequeñas que el tensor de entrada, y se traslapan gracias a
dos buffers alternados. Los números de cada fase y el speedup total están en la sección 7 y en el informe.